# Day 3 — A complete RAG loop over the AI Media Dataset

*anatoolbox try-out notebook · HSLU Computational Language Technologies capstone*

Day 2 retrieved passages. Today the loop closes: questions are rewritten, articles are
chunked, candidates are reranked, and a language model writes an answer that cites its
sources — with every step recorded.

| New today | What it gives you in Stage 3 |
|---|---|
| **Any OpenAI-compatible LLM** | Use OpenAI, a hosted provider, or a model on your own machine — configuration, not code |
| `chunk_articles_by_paragraph` | Passages sized for embedding models and prompts, each linked back to its article |
| `rewrite_query_for_retrieval` | Query expansion and decomposition — a listed Stage 3 enhancement |
| `retrieve_passages(queries_input=…)` | Retrieve with every rewrite at once, fused, with lineage |
| `rerank_passages` | Re-score candidates on their full text — another listed enhancement |
| recency and metadata filters | Time-aware retrieval for trend questions |
| `synthesize_answer` | Grounded answers with numbered citations that are *checked* |

**What stays yours:** the chunk size, the rewriting strategy, the models, the answer
instructions — and, from Day 4, how you evaluate all of it. The tools make those choices
cheap to vary; they do not make them for you.

⏱ About 5 minutes on a CPU with a small local model. Sections that need a language model
are skipped cleanly if none is configured; reranking is skipped without `sentence-transformers`.

## 0 · Setup

In [1]:
import importlib.util
import os
import platform
import time

import pandas as pd
from IPython.display import Markdown, display

import anatoolbox

pd.set_option("display.max_colwidth", 110)
HAS_EMBEDDINGS = importlib.util.find_spec("sentence_transformers") is not None
print("python     :", platform.python_version())
print("anatoolbox :", anatoolbox.__version__)
print("reranking with a cross-encoder:", "available" if HAS_EMBEDDINGS else "not installed — the rerank section will be skipped")

python     : 3.11.8
anatoolbox : 0.1.0
reranking with a cross-encoder: available


## 1 · Choose a language model

Tools that need a language model talk to any **OpenAI-compatible endpoint**. Pick one line
below, or set `ANATOOLBOX_LLM_BASE_URL`, `ANATOOLBOX_LLM_API_KEY` and `ANATOOLBOX_LLM_MODEL`
before starting Jupyter. There is no default model on purpose: which model wrote an answer
is part of the answer.

Two course-specific notes:

- **Roles.** Query rewriting asks for the `fast` role, answer writing for the `strong` role.
  Map them to different models with `configure_llm(models={"fast": ..., "strong": ...})`.
- **Stage 3 requires a different LLM to generate your Q&A pairs than the one inside your RAG
  pipeline.** Keep the two configurations clearly apart.

In [2]:
from anatoolbox.llm_client import call_llm_text, configure_llm, llm_settings

# Pick ONE (or configure through environment variables):
# configure_llm(model="gpt-4o-mini")                                                  # OpenAI — needs OPENAI_API_KEY
# configure_llm(base_url="http://localhost:11434/v1", model="qwen3:4b")                 # Ollama on your machine
# configure_llm(base_url="https://openrouter.ai/api/v1", api_key="...", model="qwen/qwen3-8b")

settings = llm_settings()
try:
    started = time.perf_counter()
    call_llm_text("Reply with one word.", "Say: ready", max_tokens=5)
    LLM_READY = True
    status = f"responding ({time.perf_counter() - started:.1f} s)"
except Exception as exc:  # no model configured, unreachable endpoint, bad key, ...
    LLM_READY = False
    status = f"not available — {type(exc).__name__}: {str(exc)[:160]}"

print("endpoint :", settings.base_url or "OpenAI")
print("models   :", settings.models or "none configured")
print("status   :", status)

endpoint : http://127.0.0.1:8765/v1
models   : {'default': 'Qwen3-0.6B'}
status   : responding (0.9 s)


## 2 · Load the dataset and pick your track

Same as Day 2: the dataset downloads from Kaggle's public API (no account needed) and is
cached in `data/`; set `AI_MEDIA_CSV` to use an existing copy. This notebook works on the
articles of **one topic track**, which is how Stage 3 is scoped.

In [3]:
import io
import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path(os.environ.get("AI_MEDIA_DATA_DIR", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/jannalipenkova/ai-media-dataset"


def find_or_download_csv():
    explicit = os.environ.get("AI_MEDIA_CSV")
    if explicit and Path(explicit).exists():
        return Path(explicit)
    cached = sorted(DATA_DIR.glob("ai_media_dataset_*.csv"))
    if cached:
        return cached[-1]
    print("Downloading the AI Media Dataset from Kaggle (~58 MB) …")
    with urllib.request.urlopen(KAGGLE_URL, timeout=180) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        name = next(n for n in archive.namelist() if n.endswith(".csv"))
        archive.extract(name, DATA_DIR)
    return DATA_DIR / name


TRACKS = {
    "Hardware & Infrastructure": r"\bgpus?\b|accelerator|nvidia|data cent(?:er|re)|semiconductor|\btsmc\b|\bchips?\b",
    "Foundation Models": r"foundation model|large language model|\bllms?\b|gpt-?\d|\bllama\b|gemini|claude|mistral|\bqwen\b",
    "Agentic Web": r"ai agents?|agentic|\bmcp\b|model context protocol|agent2agent|\ba2a\b|browser agent|computer use",
}
QUESTIONS = {
    "Agentic Web": "What security risks do autonomous browser agents create, and how are companies responding?",
    "Hardware & Infrastructure": "How are export controls affecting AI chip supply, and how are chipmakers adapting?",
    "Foundation Models": "Which open-weight foundation models were released in 2025, and how do they compare with closed models?",
}
TRACK = "Agentic Web"   # ← "Hardware & Infrastructure" | "Foundation Models" | "Agentic Web"
QUESTION = QUESTIONS[TRACK]

df = pd.read_csv(find_or_download_csv())
haystack = (df["title"] + " " + df["content"] + " " + df["tags"]).str.lower()
track_df = df[haystack.str.contains(TRACKS[TRACK], regex=True)]
track_csv = DATA_DIR / "day3_track_articles.csv"
track_df.to_csv(track_csv, index=False)
print(f"{TRACK}: {len(track_df):,} of {len(df):,} articles, {track_df['date'].min()} → {track_df['date'].max()}")
print("question:", QUESTION)

Agentic Web: 2,572 of 16,527 articles, 2024-09-11 → 2025-08-24
question: What security risks do autonomous browser agents create, and how are companies responding?


In [4]:
from anatoolbox import ToolContext, resolve_tools
from anatoolbox.memory.facade import Memory
from anatoolbox.memory.policy import RetainAllMemoryPolicy
from anatoolbox.memory.store import InMemoryRecordsetStore

anatoolbox.register_reference_tools()
memory = Memory(InMemoryRecordsetStore(), RetainAllMemoryPolicy(), project="hslu-ai-media", session_id="day3")
ctx = ToolContext(project="hslu-ai-media", session_id="day3", recordsets=memory)
ingest, chunk, rewrite, retrieve, rerank, synthesize = resolve_tools(
    ["ingest_corpus", "chunk_articles_by_paragraph", "rewrite_query_for_retrieval",
     "retrieve_passages", "rerank_passages", "synthesize_answer"]
)
print(anatoolbox.tools_by_stage())

articles = ingest.run({"path": str(track_csv), "name": "track_articles", "text_field": "content", "id_field": "Unnamed: 0"}, context=ctx)
print(f"ingested {articles['records']:,} articles → {articles['handle']}")

{'preprocess': ['chunk_articles_by_paragraph'], 'gather': ['ingest_corpus', 'rerank_passages', 'retrieve_passages', 'rewrite_query_for_retrieval'], 'enrich': ['synthesize_answer']}


ingested 2,572 articles → corpus_1


## 3 · Chunking: choosing the unit of retrieval

Embedding models and rerankers read a few hundred tokens; a whole article is far longer
(Day 2 measured a median of ~1,550 tokens). `chunk_articles_by_paragraph` splits at
paragraph boundaries and packs paragraphs to a target size. A short paragraph such as a
heading stays with the text it introduces, and no chunk exceeds `max_words`.

**Chunk size is a trade-off, not a constant.** Small chunks match specific facts precisely
but lose context; large chunks keep context but dilute the match and crowd the prompt.
The table shows what the size parameter does to this corpus.

In [5]:
from anatoolbox.corpus import get_corpus
from anatoolbox.preprocess.chunk.chunk_articles_by_paragraph import chunk_paragraphs

source = get_corpus("track_articles")
rows = []
for target in (80, 150, 300):
    sizes = [c["words"] for r in source.records for c in chunk_paragraphs(r["content"], target_size=target, max_size=2 * target, min_size=target // 5)]
    rows.append({"target_words": target, "chunks": len(sizes), "median_words": int(pd.Series(sizes).median()), "max_words": max(sizes)})
display(pd.DataFrame(rows))

,target_words,chunks,median_words,max_words
0,80,59749,65,160
1,150,33134,125,300
2,300,15704,270,600


In [6]:
chunks = chunk.run({"target_words": 150, "max_words": 300, "min_words": 30}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles → {chunks['handle']} (derived from {chunks['input_handle']})")
print("words per chunk:", chunks["words"])

chunk_corpus = get_corpus(chunks["corpus"])
first_article = chunk_corpus.records[0]["source_id"]
display(pd.DataFrame([
    {"id": r["id"], "paragraphs": f"{r['paragraph_start']}–{r['paragraph_end']}", "words": r["words"], "starts with": r["text"][:90].replace("\n", " ")}
    for r in chunk_corpus.records if r["source_id"] == first_article
]))

33,134 chunks from 2,572 articles → corpus_2 (derived from corpus_1)
words per chunk: {'min': 12, 'median': 125.0, 'max': 300}


,id,paragraphs,words,starts with
0,97824#0,0–1,80,Enterprises are looking for increasingly powerful compute to support their AI workloads an
1,97824#1,2–3,128,OCI Superclusters allow customers to choose from a wide range of NVIDIA GPUs and deploy th
2,97824#2,4–6,150,"This year, OCI will offer NVIDIA HGX H200 — connecting eight NVIDIA H200 Tensor Core GPUs"
3,97824#3,7–10,142,Companies are using NVIDIA-powered OCI Superclusters to drive AI innovation. Foundation mo
4,97824#4,11–13,119,"At Oracle CloudWorld, NVIDIA and Oracle are partnering to demonstrate three capabilities t"
5,97824#5,14–17,131,"The third demonstration illustrates how NVIDIA NIM, a set of easy-to-use inference microse"
6,97824#6,18–20,112,“ Developing a sovereign LLM allows us to offer clients a service that processes their dat
7,97824#7,21–24,140,And geospatial modeling company RSS-Hydro is demonstrating how its flood mapping platform
8,97824#8,25–26,62,Join NVIDIA at Oracle CloudWorld 2024 to learn how the companies’ collaboration is bringin


**Chunks lose context.** A chunk saying "the platform cut processing time by 30%" does not
say which platform, or when. `context_header=True` prepends each article's title and date
— a cheap, model-free step towards *contextual retrieval*, where a language model writes a
short situating summary for every chunk. Whether it helps retrieval on your questions is
something to measure, not assume.

In [7]:
headed = chunk.run({"target_words": 150, "context_header": True, "name": "track_chunks_with_header", "input": articles["handle"]}, context=ctx)
print(get_corpus(headed["corpus"]).records[1]["text"][:300])

NVIDIA and Oracle to Accelerate AI, Data Processing for Enterprises — 2024-09-11

OCI Superclusters allow customers to choose from a wide range of NVIDIA GPUs and deploy them anywhere: on premises, public cloud and sovereign cloud. Set for availability in the first half of next year, the Blackwell-b


## 4 · Rewriting the question

A compound question retrieves evidence for only one of its parts; a broad one retrieves a
little of everything. `rewrite_query_for_retrieval` turns the question into retrieval
queries — it never answers it. `decompose` writes one query per part; `expand` writes
several phrasings. `exact_terms` are rare names worth matching literally, and `time_range`
captures a period *the question* names.

In [8]:
if LLM_READY:
    for strategy in ("decompose", "expand"):
        started = time.perf_counter()
        rewrites = rewrite.run(
            {"question": QUESTION, "strategy": strategy, "collection": "AI news articles, Sep 2024 – Aug 2025"},
            context=ctx,
        )
        print(f"\n{strategy} ({time.perf_counter() - started:.1f} s, {rewrites['model']}) → {rewrites['handle']}")
        display(pd.DataFrame(rewrites["queries"]))
        print("exact terms:", rewrites["exact_terms"], "| time range:", rewrites["time_range"], "| fallback:", rewrites["fallback"])
    DECOMPOSED = memory.get("queries_1")
else:
    DECOMPOSED = None
    print("No language model configured — skipping query rewriting (see section 1).")


decompose (21.1 s, Qwen3-0.6B) → queries_1


,query,purpose
0,What security risks do autonomous browser agents create?,Find security risks associated with autonomous browser agents
1,How are companies responding to these security risks?,Find responses companies are taking to security risks


exact terms: [] | time range: {'from': None, 'to': None} | fallback: False



expand (28.4 s, Qwen3-0.6B) → queries_2


,query,purpose
0,What security risks do autonomous browser agents create?,Find security risks associated with autonomous browser agents
1,How are companies responding to security risks created by autonomous browser agents?,Find responses from companies to security risks
2,What security measures are companies implementing to address the risks created by autonomous browser agents?,Find measures taken by companies to mitigate security risks


exact terms: [] | time range: {'from': None, 'to': None} | fallback: False


## 5 · Retrieval with every rewrite, fused

Passing `queries_input` ranks the original question and each rewrite separately, then fuses
the rankings, so a passage relevant to *any* part of the question can surface. The overlap
shows how much the rewrites change what is retrieved.

Every call below names its corpus with `input`. Section 3 left two chunk variants in memory;
without an explicit handle, retrieval binds the *newest* corpus — here the header variant —
and nothing would tell you. Once you keep variants side by side, name your inputs.

In [9]:
CHUNKS = chunks["handle"]
single = retrieve.run({"query": QUESTION, "size": 30, "input": CHUNKS}, context=ctx)
fused = retrieve.run({"query": QUESTION, "size": 30, "input": CHUNKS, **({"queries_input": DECOMPOSED.handle} if DECOMPOSED else {})}, context=ctx)
single_ids = {p["id"] for p in single["passages"]}
fused_ids = {p["id"] for p in fused["passages"]}
print(f"single query: {single['handle']} | fused: {fused['handle']} derived from {memory.get(fused['handle']).derived_from}")
print(f"top-30 overlap: {len(single_ids & fused_ids)} shared, {len(fused_ids - single_ids)} new passages from the rewrites")


def passage_table(result, n=8, score="score"):
    return pd.DataFrame([
        {"rank": p["rank"], score: round(p.get(score, 0), 3), "date": p.get("date"), "article": str(p.get("title"))[:70],
         "text": p["snippet"][:100].replace("\n", " ") + " …"}
        for p in result["passages"][:n]
    ])


display(passage_table(fused))

single query: passages_1 | fused: passages_2 derived from ['corpus_2', 'queries_1']
top-30 overlap: 19 shared, 11 new passages from the rewrites


,rank,score,date,article,text
0,1,0.047,2024-10-08,Accenture's Insight on AI-Driven Cybersecurity Future,"As organizations look to unlock value with autonomous agents, security leaders are facing an even mo …"
1,2,0.039,2024-10-29,Zenity Raises $ 38 Million Series B Funding Round to Secure Agentic AI,"Recently, Zenity researchers found that the average large enterprise has nearly 80,000 AI agents, ap …"
2,3,0.039,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats,"AI is transforming software development practices, bringing new security considerations. AI coding a …"
3,4,0.037,2025-04-07,How Jeevan Jutla built an AI that thinks like a hacker ( and why that’,"Traditionally, this has been the role of vulnerability research methods like penetration testing, wh …"
4,5,0.037,2025-07-18,"Agentic AI Can’ t Be Trusted, Until It’ s Authorized","We are entering the era of agentic AI, a class of artificial intelligence that goes beyond passive p …"
5,6,0.033,2025-04-10,The AI Agent Greyscale: from limitation to hype,Information security is a great space for this kind of work. AI Agents can deeply impact monitoring …
6,7,0.033,2025-07-02,Bright Data beat Elon Musk and Meta in court — now its $ 100M AI platf,"Browser.ai represents what the company calls “ the industry’ s first unblockable, AI-native browser. …"
7,8,0.032,2025-06-03,Building and managing an agentic AI workforce,"Number two, there are a number of concerns about security and risks, from drift, hallucination, bias …"


**Time and metadata.** Trend questions care about *when*. `recency_half_life_days` makes a
passage that is one half-life older score half as much, measured from the newest date in the
corpus; `filters` restricts results by metadata such as the publishing site.

In [10]:
recent = retrieve.run({"query": QUESTION, "size": 5, "input": CHUNKS, "recency_half_life_days": 60}, context=ctx)
print("with a 60-day half-life, reference", recent["recency_reference_date"], "→ dates:", [p["date"] for p in recent["passages"]])
top_domains = track_df["domain"].value_counts().index[:2].tolist()
filtered = retrieve.run({"query": QUESTION, "size": 5, "input": CHUNKS, "filters": {"domain": top_domains}}, context=ctx)
print("filtered to", top_domains, "→", [p["domain"] for p in filtered["passages"]])

with a 60-day half-life, reference 2025-08-24 → dates: ['2025-08-11', '2025-07-18', '2025-08-11', '2025-08-12', '2025-08-07']
filtered to ['venturebeat', 'zbrain'] → ['venturebeat', 'venturebeat', 'venturebeat', 'venturebeat', 'venturebeat']


## 6 · Reranking the candidates

Retrieval is fast but coarse. `rerank_passages` reads each candidate's **full text** together
with the question and re-scores it with a cross-encoder — slow, which is why it runs on 30
candidates, not the corpus. `rank_change` shows what reranking did.

Chunking has a side effect here: several chunks of one article can fill most of the slots.
`max_per_source=2` keeps at most two passages per article.

In [11]:
if HAS_EMBEDDINGS:
    started = time.perf_counter()
    reranked = rerank.run({"keep": 6, "max_per_source": 2, "input": fused["handle"]}, context=ctx)
    print(f"reranked {reranked['candidates']} candidates with {reranked['reranker']} in {time.perf_counter() - started:.1f} s → {reranked['handle']}"
          f" ({reranked['skipped_over_source_limit']} skipped by the per-article limit)")
    display(pd.DataFrame([
        {"rank": p["rank"], "was": p["retrieval_rank"], "change": p["rank_change"], "date": p.get("date"), "article": str(p.get("title"))[:80]}
        for p in reranked["passages"]
    ]))
    ANSWER_INPUT = reranked["handle"]
else:
    print("sentence-transformers is not installed — answering from the top retrieved passages instead.")
    ANSWER_INPUT = fused["handle"]

reranked 30 candidates with cross-encoder/ms-marco-MiniLM-L-6-v2 in 8.1 s → passages_5 (0 skipped by the per-article limit)


,rank,was,change,date,article
0,1,1,0,2024-10-08,Accenture's Insight on AI-Driven Cybersecurity Future
1,2,23,21,2024-10-25,The AI Agents Are Coming — Is Your Brand Ready?
2,3,27,24,2025-07-23,Early Anthropic hire raises $ 15M to insure AI agents and help startups deploy s
3,4,3,-1,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats
4,5,7,2,2025-07-02,Bright Data beat Elon Musk and Meta in court — now its $ 100M AI platform is tak
5,6,17,11,2025-07-12,Securing the Next Frontier: Why Agentic AI Demands a New Approach to Cybersecuri


## 7 · Writing a cited answer

`synthesize_answer` labels the passages `[S1]…[Sn]`, drops duplicates, and places the most
relevant passages at the start and end of the prompt (models attend least to the middle).
Two modes:

- **grounded** — only what the sources say; say so when they are not enough.
- **blended** — may add background knowledge, marked as background and never cited.

The tool does not trust the model's citations. It reports **unknown citations** (labels that
were never provided), **uncited sources**, and **citation coverage** — the share of sentences
that carry an inline citation. None of these proves an answer faithful; Day 4 adds that
evaluation. They are cheap, objective signals, and small models trip them often.

In [12]:
answers = {}
if LLM_READY:
    for mode in ("grounded", "blended"):
        started = time.perf_counter()
        answers[mode] = synthesize.run(
            {"question": QUESTION, "mode": mode, "input": ANSWER_INPUT, "max_per_source": 2, "max_chars_per_passage": 1200, "max_tokens": 400},
            context=ctx,
        )
        answer = answers[mode]
        display(Markdown(f"### {mode} · {answer['model']} · {time.perf_counter() - started:.0f} s\n\n{answer['answer_markdown']}"))
else:
    print("No language model configured — skipping answer synthesis (see section 1).")

### grounded · Qwen3-0.6B · 35 s

Autonomous browser agents introduce new security risks, including increased data flows, new user roles, and integrations that require context-specific controls. Companies are addressing these risks by implementing comprehensive security programs with operationalized AI design and testing capabilities, as well as frameworks like SOC 2 for AI agents. Additionally, companies are using measures such as data encryption, opt-in protocols, and retention policies to protect sensitive data and operations. The rise of AI agents has also prompted a need for vigilance in managing data security and privacy concerns, as highlighted by the innovator’s dilemma.

### blended · Qwen3-0.6B · 48 s

Autonomous browser agents introduce new security risks, including increased data flows, new user roles, and integrations that require context-specific controls. Companies are addressing these risks by implementing comprehensive security programs with operationalized AI design and testing capabilities, as highlighted in [S1](https://www.accenture.com/gb-en/blogs/security/how-ai-shaping-cybersecurity-strategies). Additionally, companies like Anthropic are developing frameworks such as SOC 2 for AI agents to manage security and risk, as mentioned in [S3](https://venturebeat.com/ai/former-anthropic-exec-raises-15m-to-insure-ai-agents-and-help-startups-deploy-safely/). Bright Data also addresses these concerns by ensuring secure AI-native browser infrastructure, as noted in [S5](https://venturebeat.com/ai/bright-data-beat-elon-musk-and-meta-in-court-now-its-100m-ai-platform-is-taking-on-big-tech/). The rise of AI agents has led to a need for new approaches in cybersecurity, with recent industry data showing a threefold increase in AI agent deployments, as per [S6](https://aijourn.com/securing-the-next-frontier-why-agentic-ai-demands-a-new-approach-to-cybersecurity/). Organizations must also implement testing protocols for AI-generated code to mitigate risks, as discussed in [S4](https://aijourn.com/5-ways-to-protect-your-organization-against-evolving-digital-threats/).

In [13]:
if answers:
    display(pd.DataFrame([
        {"mode": mode, "cited": ", ".join(a["cited"]) or "—", "unknown citations": ", ".join(a["unknown_citations"]) or "—",
         "uncited sources": ", ".join(a["uncited_sources"]) or "—",
         "sentences with a citation": f"{a['statements_with_citations']} of {a['statements']}", "coverage": a["citation_coverage"]}
        for mode, a in answers.items()
    ]))
    any_answer = next(iter(answers.values()))
    display(pd.DataFrame(any_answer["sources"])[["label", "rank", "position", "date", "title", "url"]])

,mode,cited,unknown citations,uncited sources,sentences with a citation,coverage
0,grounded,—,—,"S1, S2, S3, S4, S5, S6",0 of 4,0.000
1,blended,"S1, S3, S5, S6, S4",—,S2,5 of 6,0.833


,label,rank,position,date,title,url
0,S1,1,1,2024-10-08,Accenture's Insight on AI-Driven Cybersecurity Future,https://www.accenture.com/gb-en/blogs/security/how-ai-shaping-cybersecurity-strategies
1,S2,2,6,2024-10-25,The AI Agents Are Coming — Is Your Brand Ready?,https://medium.com/ipg-media-lab/the-ai-agents-are-coming-is-your-brand-ready-f6c493b1ccb0
2,S3,3,2,2025-07-23,Early Anthropic hire raises $ 15M to insure AI agents and help startups deploy safely,https://venturebeat.com/ai/former-anthropic-exec-raises-15m-to-insure-ai-agents-and-help-startups-deploy-s...
3,S4,4,5,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats,https://aijourn.com/5-ways-to-protect-your-organization-against-evolving-digital-threats/
4,S5,5,3,2025-07-02,Bright Data beat Elon Musk and Meta in court — now its $ 100M AI platform is taking on Big Tech,https://venturebeat.com/ai/bright-data-beat-elon-musk-and-meta-in-court-now-its-100m-ai-platform-is-taking...
5,S6,6,4,2025-07-12,Securing the Next Frontier: Why Agentic AI Demands a New Approach to Cybersecurity,https://aijourn.com/securing-the-next-frontier-why-agentic-ai-demands-a-new-approach-to-cybersecurity/


**Reading these results.** With a small model, expect trouble — and expect it to vary
between runs. In runs of this notebook, Qwen3-0.6B has written grounded answers with no
citation at all, collected citations at the end of a paragraph instead of after each
sentence, written a source list it was told not to write, and made statements the checks
cannot confirm the sources support. None of that is a reason to hide the model's output —
it is exactly what your evaluation has to catch, and why model choice is a Stage 3 design
decision.

## 8 · Lineage: the whole loop on one page

Every step was recorded with its settings and the handles it was derived from. When an
evaluation in Day 4 flags an answer, this is how you find out whether the chunking, the
rewrite, the retrieval, the reranking, or the model was responsible.

In [14]:
display(pd.DataFrame([
    {"handle": r.handle, "produced by": r.produced_by, "count": r.count, "derived from": ", ".join(r.derived_from) or "—",
     "settings": {k: v for k, v in r.args.items() if k in ("strategy", "target_words", "context_header", "keep", "max_per_source", "mode", "model", "reranker", "recency_half_life_days", "filters")}}
    for r in sorted(memory.list(), key=lambda r: r.created_at)
]))

,handle,produced by,count,derived from,settings
0,corpus_1,ingest_corpus,2572,—,{}
1,corpus_2,chunk_articles_by_paragraph,33134,corpus_1,"{'target_words': 150, 'context_header': False}"
2,corpus_3,chunk_articles_by_paragraph,33134,corpus_1,"{'target_words': 150, 'context_header': True}"
3,queries_1,rewrite_query_for_retrieval,2,—,"{'strategy': 'decompose', 'model': 'Qwen3-0.6B'}"
4,queries_2,rewrite_query_for_retrieval,3,—,"{'strategy': 'expand', 'model': 'Qwen3-0.6B'}"
5,passages_1,retrieve_passages,30,corpus_2,{'strategy': 'sparse'}
6,passages_2,retrieve_passages,30,"corpus_2, queries_1",{'strategy': 'sparse'}
7,passages_3,retrieve_passages,5,corpus_2,"{'strategy': 'sparse', 'recency_half_life_days': 60.0}"
8,passages_4,retrieve_passages,5,corpus_2,"{'strategy': 'sparse', 'filters': {'domain': ['venturebeat', 'zbrain']}}"
9,passages_5,rerank_passages,6,passages_2,"{'keep': 6, 'max_per_source': 2, 'reranker': 'cross-encoder/ms-marco-MiniLM-L-6-v2'}"


## What to take away

- **Every Stage 3 enhancement the brief lists is now an argument:** chunk size, query
  rewriting, reranking, retrieval filtering. Comparing variants means changing a setting and
  keeping the lineage, not rebuilding the pipeline.
- **Model choice is visible and swappable.** Any OpenAI-compatible endpoint works, the model
  used is recorded with every answer, and roles let rewriting and answering use different models.
- **Citations are checked, not trusted.** Unknown labels, uncited sources and citation coverage
  are objective signals — and a small local model shows why you need them.

**Next — Day 4: evaluation.** Retrieval metrics (precision, recall, MRR), answer faithfulness
and relevance with an LLM as judge, and a Q&A dataset built with a *different* model — plus
the hand-off that brings your Stage 1 knowledge graph into retrieval.